# Direct MNA / SSN component benchmark

This notebook records small-grid comparisons between each component's original subcomponent implementation and its optional direct/optimized constructor. Legacy constructors remain the default. Runs use a 0.5 s EMT simulation; most use Δt = 50 μs (10,000 steps), and the DC π-line uses Δt = 10 μs (50,000 steps). Timings are arithmetic means of five runs unless marked provisional.

The trace error is the maximum absolute difference over logged values; values are rounded by the CSV logger. Scaled error is max absolute error divided by the largest legacy trace magnitude. The direct transient-stability generator has one correctness run so far and is intentionally marked provisional for timing. This is a component-level microbenchmark, not a prediction of large-grid speedup.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rows = [
    ("EMT Ph3 RxLine", 0.071879, 0.065402, 0.0, 0.0, "5 runs"),
    ("EMT Ph3 NetworkInjection", 0.067671, 0.054320, 0.0, 0.0, "5 runs"),
    ("EMT Ph3 Shunt", 0.059923, 0.059201, 0.0, 0.0, "5 runs"),
    ("EMT Ph3 PiLine", 0.068398, 0.059096, 1.00001e-6, 7.41167e-12, "5 runs"),
    ("EMT Ph3 RXLoad", 0.062459, 0.060071, 0.0, 0.0, "5 runs"),
    ("EMT Ph3 Transformer", 0.066036, 0.061276, 1.29e-4, 1.29e-9, "5 runs"),
    ("EMT Ph3 HalfDecouplingLine", 0.113463, 0.104667, 0.0, 0.0, "5 runs"),
    ("EMT DC PiLine", 0.254789, 0.213369, 1.0e-6, 4.48061e-11, "5 runs"),
    ("EMT Ph1 PiLine", 0.048272, 0.040098, 1.0e-6, 8.03574e-12, "5 runs"),
    ("EMT Ph1 VoltageSourceRamp", 0.035816, 0.036327, 0.0, 0.0, "5 runs"),
    (
        "EMT Ph3 SynchronGeneratorIdeal",
        0.053162,
        0.052868,
        1.00001e-6,
        1.22475e-11,
        "5 runs",
    ),
    (
        "EMT Ph3 SynchronGeneratorTrStab",
        0.071287,
        0.072292,
        0.0,
        0.0,
        "1 run — provisional",
    ),
    (
        "EMT Ph3 GFL (no transformer)",
        1.524383,
        1.376703,
        13.458819,
        0.03204,
        "1 run — provisional",
    ),
]
df = pd.DataFrame(
    rows,
    columns=[
        "Component",
        "Legacy mean (s)",
        "Direct mean (s)",
        "Max abs trace error",
        "Scaled trace error",
        "Timing sample",
    ],
)
df["Speedup (%)"] = (
    100 * (df["Legacy mean (s)"] - df["Direct mean (s)"]) / df["Legacy mean (s)"]
)
display(
    df.style.format(
        {
            "Legacy mean (s)": "{:.6f}",
            "Direct mean (s)": "{:.6f}",
            "Max abs trace error": "{:.3g}",
            "Scaled trace error": "{:.3g}",
            "Speedup (%)": "{:+.2f}%",
        }
    )
)

In [ ]:
plot_df = df[df["Timing sample"] == "5 runs"].set_index("Component")
ax = plot_df[["Legacy mean (s)", "Direct mean (s)"]].plot.bar(
    figsize=(13, 5), color=["#6b7280", "#2563eb"]
)
ax.set_ylabel("Wall time (s)")
ax.set_title("EMT component legacy vs direct implementation — five-run means")
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()

## Interpretation and coverage

Direct mode improves most passive EMT cases in this small setup, while the ramp source and ideal generator are effectively neutral. HalfDecouplingLine's direct path is close to neutral and sensitive to run-to-run noise. Use the per-run average and numerical trace check together; do not infer a grid-scale result from these short runs.

The GFL result is a 10 s controller-driven case and remains provisional after one run; its largest absolute trace difference is 13.46 in a controller/source-reference signal (about 3.2% on the largest shared logged signal scale), so it needs repeated timing and further long-horizon accuracy review. The other EMT inverter/controller composites and DP classes are still pending. Their results will be added here only after each direct constructor preserves its existing default path and passes a representative numerical comparison.